# Add a PDF facsimile and measure zones to one MEI file

**Workflow 1 — editing music with the MEI data format.** This notebook starts with one clean MEI file, rasterizes a multi-page PDF, reviews page roles and systems, detects measure boxes, aligns them, and writes surfaces, zones and `@facs` into a new MEI file. An optional last step rewrites page and system breaks to match the print.

The example is `test_corpus/Buxtehude-DdT11-app01-suite.mei`: Buxtehude's Suite (Denkmäler deutscher Tonkunst vol. 11, appendix no. 1), 89 measures across four printed pages, with the IIIF `<facsimile>` removed so the tutorial starts clean. The PDF is Bayerische Staatsbibliothek `bsb00023199`, images 0170–0173 (printed pp. 160–163); viewer <https://www.digitale-sammlungen.de/en/view/bsb00023199?page=170>.

Network calls, detector upload and writes stay off until you set the flags below. Generated files go to `converted_mei/pdf_tutorial/` (gitignored). The original in `test_corpus/` is not overwritten.

Continue afterwards with [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb). See also [Link MEI to a PDF facsimile](../docs/guides/pdf-facsimiles.md).


In [ ]:
# Cloud Jupyter: install CAMAT and copy notebooks/test_corpus if they are not already here.
try:
    import setup_camat
except ModuleNotFoundError:
    pass
try:
    from camat.notebook_workspace import prepare_notebook
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "camat"])
    from camat.notebook_workspace import prepare_notebook

prepare_notebook()

# You can leave this cell unchanged.
try:
    from camat import display_path, resolve_repo_path, sha256_file
except ModuleNotFoundError:
    import setup_camat
    from camat import display_path, resolve_repo_path, sha256_file

from camat.pdf_facsimile import (
    build_measure_alignment,
    detect_measure_boxes,
    draw_alignment_overlays,
    integrate_pdf_facsimile,
    render_pdf_pages,
    write_page_breaks,
)

import json
import re
import shutil
from datetime import date
from pathlib import Path

def _show_image(path, *, width=480):
    """Show an image in Jupyter; print the path under headless IPython."""
    path = Path(path)
    print(display_path(path))
    try:
        from IPython import get_ipython
        ip = get_ipython()
    except ImportError:
        ip = None
    if ip is None or getattr(ip, "kernel", None) is None:
        return
    from IPython.display import Image, display
    display(Image(filename=str(path), width=width))


## 1. Configure the job

Set `SOURCE_MEI` to a clean MEI file (no `<facsimile>` yet) and `PDF_PATH` to the local PDF. With the defaults, **Run All** finishes without writing anything: each step prints that it was skipped.

Review the plan in the next cells, then set `RUN_PDF_INTEGRATION = True`. The BSB download and detector upload stay off unless you also set `ACCEPT_BSB_TERMS` / `ALLOW_DETECTOR_UPLOAD`. Approve the alignment only with a non-empty `REVIEWER` and `APPROVE_REVIEW = True`.


In [ ]:
SOURCE_MEI = "test_corpus/Buxtehude-DdT11-app01-suite.mei"
PDF_PATH = "converted_mei/pdf_tutorial/source/bsb00023199_0170-0173.pdf"
TARGET_DIR = "converted_mei/pdf_tutorial"
ACCEPT_BSB_TERMS = False
ALLOW_DETECTOR_UPLOAD = False
RUN_PDF_INTEGRATION = False
APPROVE_REVIEW = False
REVIEWER = ""

OVERWRITE_STAGED_MEI = False
DPI = 300
SOURCE_ID = "bsb00023199"


## 2. Review the plan

This cell only reads the source and prints the paths that would be written. It does not download or change files.

The **measure detector** is Edirom's DOMD service (same endpoint as the IIIF tutorial): a neural-network model that finds measure boxes on the page *picture*. It does not read the MEI. CAMAT uploads the page image only when `ALLOW_DETECTOR_UPLOAD = True`, receives bounding boxes, and later writes them as `<zone type="measure">` elements.


In [ ]:
source_mei = resolve_repo_path(SOURCE_MEI)
pdf_path = resolve_repo_path(PDF_PATH)
target_dir = resolve_repo_path(TARGET_DIR)
pages_dir = target_dir / "pages"
pages_json = pages_dir / "pages.json"
boxes_dir = pages_dir / "boxes"
alignment_json = pages_dir / "alignment.json"
overlays_dir = target_dir / "overlays"
mei_dir = target_dir / "mei"
staged_mei = mei_dir / source_mei.name
integrated_mei = mei_dir / f"{source_mei.stem}_pdf_facs.mei"
breaks_mei = mei_dir / f"{source_mei.stem}_pdf_facs_breaks.mei"

if not source_mei.is_file():
    raise FileNotFoundError(source_mei)

print(f"Source MEI:   {display_path(source_mei)}")
print(f"PDF:          {display_path(pdf_path)}")
print(f"Target dir:   {display_path(target_dir)}")
print(f"Pages:        {display_path(pages_dir)}")
print(f"Alignment:    {display_path(alignment_json)}")
print(f"Overlays:     {display_path(overlays_dir)}")
print(f"Staged MEI:   {display_path(staged_mei)}")
print(f"Integrated:   {display_path(integrated_mei)}")
print(f"With breaks:  {display_path(breaks_mei)}")
print(f"Run enabled:  {RUN_PDF_INTEGRATION}")


## 3. Check Poppler

`render_pdf_pages` needs Poppler's `pdftoppm` (and `pdfinfo`) on `PATH`.


In [ ]:
import shutil

if shutil.which("pdftoppm") and shutil.which("pdfinfo"):
    print("Poppler found:", shutil.which("pdftoppm"))
else:
    print("Poppler is missing. Install it, then re-run this cell:")
    print("  Debian/Ubuntu:  sudo apt install poppler-utils")
    print("  Homebrew:       brew install poppler")
    print("  conda-forge:    conda install -c conda-forge poppler")


## 4. Get the PDF

The scans are **CC BY-NC-SA 4.0** (Bayerische Staatsbibliothek). Do not commit or redistribute them with CAMAT.

**Default:** download BSB's PDF for images ("Bild") 170–173 yourself and save it at `PDF_PATH`. From the [viewer page](https://www.digitale-sammlungen.de/en/view/bsb00023199?page=170), use the download form for those image numbers.

Only with `ACCEPT_BSB_TERMS = True` does this notebook fetch the file: that answers BSB's rights notice (`xdfz=2` = "Ja") and downloads the one `/pdf/<digits>bsb00023199.pdf` link from the form response. An existing PDF is never overwritten.

The PDF has **5 pages**: an MDZ cover with metadata, then the four scans (printed 160–163). Every download differs in the PDF trailer `/ID`, so its sha256 changes even when the rendered pages are identical. `render_pdf_pages` keys reuse on the PDF hash — keep your file. A re-download is a new source, and rendering refuses to overwrite the old `pages/`. Move `pages/` aside, render again, then copy the old `pages/boxes/` folder into the new `pages/`. Measure boxes are tied to the **page-image** hash, and a fresh download renders to the same pixels, so the saved detector responses and the bar-10 correction below are reused without a new upload. Deleting `pages/` deletes them too.


In [ ]:
BSB_FORM_URL = (
    "https://download.digitale-sammlungen.de/BOOKS/download.pl"
    "?id=00023199&ersteseite=170&letzteseite=173&xdfz=2&vers=d&nr=&abschicken=ja"
    "&captcha_zeit=00023199"
)
BSB_HOST = "https://download.digitale-sammlungen.de"
PDF_HREF = re.compile(r'href=[\'"](/pdf/\d+bsb00023199\.pdf)[\'"]')


def pdf_link_from_bsb_html(html: str) -> str:
    matches = PDF_HREF.findall(html)
    if len(matches) != 1:
        raise ValueError(f"expected one /pdf/<digits>bsb00023199.pdf link, found {len(matches)}")
    return matches[0]


if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif pdf_path.is_file():
    print(f"Using existing PDF: {display_path(pdf_path)}")
    print(f"sha256: {sha256_file(pdf_path)}")
elif not ACCEPT_BSB_TERMS:
    print("Skipped download. Save BSB's PDF for images 170–173 at:")
    print(f"  {display_path(pdf_path)}")
    print("Or set ACCEPT_BSB_TERMS = True to fetch it (CC BY-NC-SA 4.0; do not redistribute with CAMAT).")
else:
    import urllib.request

    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(BSB_FORM_URL, timeout=120) as response:
        html = response.read().decode("utf-8", errors="replace")
    href = pdf_link_from_bsb_html(html)
    url = BSB_HOST + href
    print(f"Downloading {url}")
    with urllib.request.urlopen(url, timeout=300) as response:
        data = response.read()
    if not data.startswith(b"%PDF-"):
        raise ValueError(f"{url} did not return a PDF; download it by hand instead (see above).")
    pdf_path.write_bytes(data)
    print(f"Wrote {display_path(pdf_path)} ({len(data)} bytes)")
    print(f"sha256: {sha256_file(pdf_path)}")


## 5. Render the pages

Rasterize every PDF page at 300 dpi. A second run with the same PDF and renderer reuses the images.


In [ ]:
if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not pdf_path.is_file():
    print(f"Skipped. PDF missing at {display_path(pdf_path)}.")
else:
    manifest = render_pdf_pages(pdf_path, pages_dir, dpi=DPI, source_id=SOURCE_ID)
    print(f"pages.json: {display_path(pages_json)}")
    print(f"pages: {manifest['page_count']}, dpi={manifest['dpi']}, renderer={manifest['renderer']}")
    for page in manifest["pages"]:
        print(f"  {page['page_id']}: {page['image']} {page['width']}x{page['height']} role={page['role']}")


## 6. Review the pages

Fill in the review fields of `pages.json`. For this print:

| Field | Meaning here |
| --- | --- |
| `role` | PDF page 1 is the MDZ `cover`; pages 2–5 are `music` |
| `printed_page` | `"160"` … `"163"` as printed (strings) |
| `mdiv` | `"1"` — the Suite is one movement in the MEI |
| `systems` | measure `@n` labels per system, top to bottom; the print has no bar numbers, so the MEI's continuous 1–89 is used; no split bars |

Systems for this example: p.160 `1–4 | 5–9 | 10–14`; p.161 `15–18 | 19–25 | 26–33 | 34–40`; p.162 `41–47 | 48–55 | 56–63 | 64–70`; p.163 `71–74 | 75–79 | 80–84 | 85–89`.


In [ ]:
PAGE_REVIEW = {
    "bsb00023199-p0001": {"role": "cover"},
    "bsb00023199-p0002": {
        "role": "music",
        "printed_page": "160",
        "mdiv": "1",
        "systems": [
            ["1", "2", "3", "4"],
            ["5", "6", "7", "8", "9"],
            ["10", "11", "12", "13", "14"],
        ],
    },
    "bsb00023199-p0003": {
        "role": "music",
        "printed_page": "161",
        "mdiv": "1",
        "systems": [
            ["15", "16", "17", "18"],
            ["19", "20", "21", "22", "23", "24", "25"],
            ["26", "27", "28", "29", "30", "31", "32", "33"],
            ["34", "35", "36", "37", "38", "39", "40"],
        ],
    },
    "bsb00023199-p0004": {
        "role": "music",
        "printed_page": "162",
        "mdiv": "1",
        "systems": [
            ["41", "42", "43", "44", "45", "46", "47"],
            ["48", "49", "50", "51", "52", "53", "54", "55"],
            ["56", "57", "58", "59", "60", "61", "62", "63"],
            ["64", "65", "66", "67", "68", "69", "70"],
        ],
    },
    "bsb00023199-p0005": {
        "role": "music",
        "printed_page": "163",
        "mdiv": "1",
        "systems": [
            ["71", "72", "73", "74"],
            ["75", "76", "77", "78", "79"],
            ["80", "81", "82", "83", "84"],
            ["85", "86", "87", "88", "89"],
        ],
    },
}

if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not pages_json.is_file():
    print(f"Skipped. Missing {display_path(pages_json)}; render first.")
else:
    manifest = json.loads(pages_json.read_text(encoding="utf-8"))
    for page in manifest["pages"]:
        review = PAGE_REVIEW[page["page_id"]]
        page["role"] = review["role"]
        if review["role"] == "music":
            page["printed_page"] = review["printed_page"]
            page["mdiv"] = review["mdiv"]
            page["systems"] = review["systems"]
        else:
            page["printed_page"] = None
            page.pop("mdiv", None)
            page.pop("systems", None)
    pages_json.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(f"Updated {display_path(pages_json)}")
    for page in manifest["pages"]:
        systems = page.get("systems")
        n = sum(len(s) for s in systems) if systems else 0
        print(f"  {page['page_id']}: role={page['role']} printed={page.get('printed_page')} measures={n}")


## 7. Detect measure boxes

Uploads send the music-page PNGs to Edirom's DOMD service (same as the IIIF tutorial). Cached responses under `pages/boxes/` are reused by page-image hash without network access.


In [ ]:
if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not pages_json.is_file():
    print(f"Skipped. Missing {display_path(pages_json)}.")
else:
    try:
        records = detect_measure_boxes(pages_json, allow_upload=ALLOW_DETECTOR_UPLOAD)
    except FileNotFoundError as exc:
        print(exc)
        print("Set ALLOW_DETECTOR_UPLOAD = True to send the music pages to the detector,")
        print("or place saved boxes under pages/boxes/ (one <page_id>.json per music page).")
    else:
        for record in records:
            print(
                f"{record['page_id']}: {record['measures']} boxes "
                f"({record['source']}; uploaded={record['uploaded']})"
            )


## 8. Align measures — and fix one detector miss

`build_measure_alignment` groups boxes into systems and compares counts with `pages.json`. On this file the detector misses bar 10 on p.160: system 3 starts with a repeat and an upbeat that the MEI encodes as its own measure (`measure n="10" left="rptstart"`), so DOMD returns boxes only for 11–14 and the gate reports `actual [4, 5, 4], expected [4, 5, 5]`.

The cell catches that message, shows the raw boxes (system-3 band y ≈ 2540–3560 on the 2860×3822 raster), appends one manual box for bar 10 if none already sits left of the first barline (`x < 400`), and aligns again. Detector boxes stay in their original order; `.run.json` is not touched (the page becomes `detector-edited`).


In [ ]:
BAR10_BOX = {"ulx": 180, "uly": 2540, "lrx": 530, "lry": 3560}
PAGE2_ID = "bsb00023199-p0002"
PAGE2_BOXES = boxes_dir / f"{PAGE2_ID}.json"


def ensure_bar10_box(path: Path) -> str:
    payload = json.loads(path.read_text(encoding="utf-8"))
    measures = payload["measures"]
    system3 = [
        box for box in measures
        if box["uly"] >= 2500 and box["lry"] <= 3600
    ]
    if any(box["ulx"] < 400 for box in system3):
        return f"bar 10 already present ({len(measures)} boxes); not adding"
    measures.append(dict(BAR10_BOX))
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    return f"appended bar 10 box; now {len(measures)} boxes"


def show_raw_boxes(path: Path, image_path: Path) -> None:
    from PIL import Image as PILImage, ImageDraw

    payload = json.loads(path.read_text(encoding="utf-8"))
    with PILImage.open(image_path) as source:
        image = source.convert("RGB")
    draw = ImageDraw.Draw(image, "RGBA")
    for box in payload["measures"]:
        draw.rectangle(
            [box["ulx"], box["uly"], box["lrx"], box["lry"]],
            outline=(200, 40, 40, 230),
            width=3,
        )
    preview = path.with_suffix(".preview.png")
    image.save(preview)
    _show_image(preview)


if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not pages_json.is_file():
    print(f"Skipped. Missing {display_path(pages_json)}.")
elif missing := [
    page["page_id"]
    for page in json.loads(pages_json.read_text(encoding="utf-8"))["pages"]
    if page.get("role") == "music" and not (boxes_dir / f"{page['page_id']}.json").is_file()
]:
    print(f"Skipped. No measure boxes yet for {', '.join(missing)}; run the detection step first.")
else:
    try:
        build_measure_alignment(pages_json, output=alignment_json)
    except ValueError as exc:
        print("Alignment gate:")
        print(exc)
        # Repair only the known miss; any other gate failure (or a stale hash) is the reader's to review.
        known_miss = f"{PAGE2_ID} (PDF page 2, printed page 160): actual [4, 5, 4], expected [4, 5, 5]"
        if str(exc).splitlines()[1:] == [known_miss] and PAGE2_BOXES.is_file():
            show_raw_boxes(PAGE2_BOXES, pages_dir / "page-0002.png")
            print(ensure_bar10_box(PAGE2_BOXES))
            records = detect_measure_boxes(pages_json, allow_upload=False)
            for record in records:
                if record["page_id"] == PAGE2_ID:
                    print(f"re-check {PAGE2_ID}: {record['measures']} boxes ({record['source']})")
        else:
            raise
    alignment_path = build_measure_alignment(pages_json, output=alignment_json)
    data = json.loads(alignment_path.read_text(encoding="utf-8"))
    print(f"alignment: {display_path(alignment_path)} ({len(data['regions'])} regions)")


## 9. Draw alignment overlays

One labelled PNG per music page. Look for missing/extra boxes, wrong system grouping, and movement starts.


In [ ]:
if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not alignment_json.is_file():
    print(f"Skipped. Missing {display_path(alignment_json)}.")
else:
    paths = draw_alignment_overlays(alignment_json, overlays_dir)
    for path in paths:
        _show_image(path)


## 10. Approve the review

Set each music page's `review` entry to `approved` only when `APPROVE_REVIEW` is true and `REVIEWER` is non-empty. Entries that are already approved are left unchanged so a re-run rewrites nothing.


In [ ]:
if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not APPROVE_REVIEW or not REVIEWER.strip():
    print("Skipped. Set APPROVE_REVIEW = True and REVIEWER to a non-empty name.")
elif not alignment_json.is_file():
    print(f"Skipped. Missing {display_path(alignment_json)}.")
else:
    data = json.loads(alignment_json.read_text(encoding="utf-8"))
    today = date.today().isoformat()
    changed = 0
    for page_id, entry in data.get("review", {}).items():
        if entry.get("status") == "approved":
            continue
        entry["status"] = "approved"
        entry["reviewer"] = REVIEWER.strip()
        entry["date"] = today
        changed += 1
    if changed:
        alignment_json.write_text(json.dumps(data, indent=2) + "\n", encoding="utf-8")
    print(f"Approved {changed} page(s); reviewer={REVIEWER!r} date={today}")


## 11. Stage the MEI and number the movement

Copy the fixture into `mei/` and set `n="1"` on the single unnumbered `<mdiv>`. Integration pairs measures by `(mdiv/@n, measure/@n)`; Verovio-converted MEI often omits `mdiv/@n`, and `integrate_pdf_facsimile` then stops with "No linked movements".

A second run keeps an existing staged copy when its bytes match what this cell would write. Set `OVERWRITE_STAGED_MEI = True` to replace an edited copy.


In [ ]:
def number_lone_mdiv(text: str) -> str:
    matches = list(re.finditer(r"<mdiv\b[^>]*>", text))
    lacking = [m for m in matches if not re.search(r'\sn="', m.group(0))]
    if len(lacking) != 1:
        raise ValueError(f"expected exactly one <mdiv> without @n, found {len(lacking)}")
    tag = lacking[0].group(0)
    if tag.endswith("/>"):
        raised = tag[:-2] + ' n="1"/>'
    else:
        raised = tag[:-1] + ' n="1">'
    return text[: lacking[0].start()] + raised + text[lacking[0].end() :]


if not RUN_PDF_INTEGRATION:
    print("Skipped.")
else:
    desired = number_lone_mdiv(source_mei.read_text(encoding="utf-8"))
    mei_dir.mkdir(parents=True, exist_ok=True)
    if staged_mei.is_file():
        existing = staged_mei.read_text(encoding="utf-8")
        if existing == desired:
            print(f"Staged copy already up to date: {display_path(staged_mei)}")
        elif not OVERWRITE_STAGED_MEI:
            raise FileExistsError(
                f"{display_path(staged_mei)} differs from the numbered fixture copy. "
                "Set OVERWRITE_STAGED_MEI = True to replace it."
            )
        else:
            staged_mei.write_text(desired, encoding="utf-8")
            print(f"Overwrote staged copy: {display_path(staged_mei)}")
    else:
        staged_mei.write_text(desired, encoding="utf-8")
        print(f"Staged {display_path(source_mei)} -> {display_path(staged_mei)} (mdiv @n=1)")


## 12. Integrate the facsimile

Writes `<facsimile>`, one `<surface>` per music page, zones, and `measure/@facs` / `pb/@facs`. Existing MusicXML layout `<pb>` elements that are not at a printed page start are listed in the report — that motivates the optional break rewrite next.


In [ ]:
if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not staged_mei.is_file() or not alignment_json.is_file():
    print("Skipped. Stage the MEI and finish alignment first.")
else:
    report = integrate_pdf_facsimile(staged_mei, pages_json, alignment_json, integrated_mei)
    print(json.dumps(report, indent=2))
    print(f"Wrote {display_path(integrated_mei)}")


## 13. Optional: rewrite page and system breaks

`write_page_breaks` replaces the linked movement's `<pb>` / `<sb>` with the reviewed layout from `pages.json` (one break before the first measure of each printed system) and links each `<pb>` to its surface.


In [ ]:
if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not integrated_mei.is_file():
    print(f"Skipped. Missing {display_path(integrated_mei)}.")
else:
    report = write_page_breaks(integrated_mei, pages_json, breaks_mei)
    print(json.dumps(report, indent=2))
    print(f"Wrote {display_path(breaks_mei)}")


## 14. Check the result

Same checks as the guide's §8: schema, Verovio load/render and CAMAT parse. Missing optional validators (for example `saxonche` for Schematron) are reported as skipped instead of raising.

The CAMAT parse uses the music21 backend here: Verovio 6.3.0 loads and renders this file, but crashes when it exports it to MIDI, which the default `verovio` backend needs.


In [ ]:
if not RUN_PDF_INTEGRATION:
    print("Skipped.")
elif not breaks_mei.is_file():
    print(f"Skipped. Missing {display_path(breaks_mei)}.")
else:
    import subprocess
    import sys
    import tempfile

    with tempfile.TemporaryDirectory(prefix="camat-pdf-tutorial-validate-") as tmp:
        cmd = [
            sys.executable,
            "-m",
            "camat.mei_validation",
            str(breaks_mei),
            "--output-dir",
            tmp,
            "--verovio",
            "--camat-parse",
            "--parsing-backend",
            "music21",
        ]
        run = subprocess.run(
            cmd,
            cwd=str(resolve_repo_path(".")),
            capture_output=True,
            text=True,
        )
        run_path = Path(tmp) / "run.json"
        if run_path.is_file():
            record = json.loads(run_path.read_text(encoding="utf-8"))
            for name, entry in record.get("executions", {}).items():
                status = entry.get("status")
                reason = entry.get("reason")
                line = f"{name}: {status}"
                if reason:
                    line += f" ({reason})"
                print(line)
            print(f"conformance: {record.get('conformance')}")
            if run.returncode < 0:
                print(
                    f"(validator process aborted by signal {-run.returncode}; "
                    "later checks may be incomplete)"
                )
            elif run.returncode not in (0, 1):
                print(f"(validator exit code {run.returncode})")
        else:
            print(f"validation did not write a report (exit {run.returncode})")
            if run.stderr.strip():
                print(run.stderr.strip()[-500:])


## 15. View the result

Open [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb) and set:

```python
MEI_SOURCE = "converted_mei/pdf_tutorial/mei/Buxtehude-DdT11-app01-suite_pdf_facs_breaks.mei"
```

Use `verovio_breaks="encoded"` so Verovio follows the print's system breaks written above.


## Notes for the next file

1. Change `SOURCE_MEI` and pair it with a PDF at `PDF_PATH` (or enable `ACCEPT_BSB_TERMS` for this BSB example).
2. Review the plan, then set `RUN_PDF_INTEGRATION = True`.
3. Enable `ALLOW_DETECTOR_UPLOAD` only when boxes are missing and upload is allowed for the source.
4. After reviewing overlays, set `APPROVE_REVIEW = True` and a non-empty `REVIEWER`.
5. If the staged MEI was edited by hand and should be replaced, set `OVERWRITE_STAGED_MEI = True`.
6. Keep your PDF: a re-download changes the trailer `/ID` and counts as a new source for `render_pdf_pages`.
7. Inspect the `_breaks` file in [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb), then continue with consistency checks.
